# ECSE 316 Assignment 1: Reliable records over TCP

A laboratory is building a collection service for binary records.
Clients may send a short status record, a larger measurement block, or
any other sequence of bytes. The collector acts as the server: it
must recover each complete record, detect accidental corruption,
acknowledge valid data, and close the session cleanly.

TCP delivers an ordered stream of bytes, but it does not preserve the
boundaries between application records. In this assignment you will
build the small protocol layer that the collection service needs.


## Assignment path

We will develop the collector in six stages:

1. create a TCP connection and identify the role of each socket;
2. observe how TCP's byte stream behaves;
3. make partial socket reads and writes reliable;
4. turn the stream into framed records;
5. implement the application-protocol conversation; and
6. measure the latency/goodput tradeoff of changing record size.

By the end, you should be able to:

- explain how `bind`, `listen`, `connect`, and `accept` establish a
  TCP connection;
- distinguish a listening socket from a connected socket;
- explain why TCP applications need framing;
- implement robust stream I/O and a binary frame format;
- describe a protocol in terms of syntax, meaning, and sequencing; and
- connect application design choices to measured performance.

## Working with the assignment folder

1. Download and unzip the supplied `ECSE316_Assignment1` folder.
2. Upload the complete folder to `My Drive/ECSE316/` so its path is
   `My Drive/ECSE316/ECSE316_Assignment1/`.
3. Open this notebook from Drive with Google Colab.
4. When instructed, use Colab's Files pane to open the named Python
   file. Save your changes, then run the following check cell.


In [ ]:
from pathlib import Path
import sys
import importlib
import types

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    # Change this value only if you uploaded the folder elsewhere.
    FOLDERNAME = "ECSE316/ECSE316_Assignment1"
    drive_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
    ]
    candidates = [
        *(root / FOLDERNAME for root in drive_roots),
        *(root / "ECSE316_Assignment1" for root in drive_roots),
    ]
else:
    candidates = [
        Path.cwd(),
        Path.cwd() / "ECSE316_Assignment1",
    ]

ASSIGNMENT_ROOT = next(
    (
        candidate
        for candidate in candidates
        if (candidate / "ecse316_a1" / "support.py").is_file()
    ),
    None,
)
if ASSIGNMENT_ROOT is None:
    checked = "\n".join(f"  - {path}" for path in candidates)
    raise FileNotFoundError(
        "Could not find the extracted ECSE316_Assignment1 folder.\n"
        "Confirm that it contains ecse316_a1/support.py.\n"
        "If it is elsewhere in Drive, update FOLDERNAME in this cell.\n"
        f"Locations checked:\n{checked}"
    )

if str(ASSIGNMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(ASSIGNMENT_ROOT))
importlib.invalidate_caches()

# Compatibility shim for older IPython autoreload on Python 3.12+.
sys.modules["imp"] = types.ModuleType("imp")
import imp
imp.reload = importlib.reload

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

from ecse316_a1 import support
print("Assignment folder:", ASSIGNMENT_ROOT)
support.show_environment()


## Team information


In [ ]:
# Keep one entry per team member (one to three total).
TEAM_NUMBER = "XX"
TEAM_MEMBERS = [("Name", "Student ID"), ("Name", "Student ID"), ("Name", "Student ID")]


## Assessment

| Component | Weight |
|---|---:|
| `send_all` and `receive_exactly` | 20% |
| `encode_frame`, `send_frame`, and `receive_frame` | 30% |
| `handle_client` and complete TCP exchange | 15% |
| Experiment and evidence | 20% |
| Design questions | 10% |
| Completion and clarity | 5% |

Visible checks are examples, not the complete specification. Additional
tests may cover documented fragmentation, EOF, invalid-frame, and
lifecycle cases. Absolute speed is not graded.


## Part 1 — From processes to a TCP connection

A **socket** is an operating-system object through which a process uses
the network. In Python, a `socket.socket` object is the program's
handle to that object. For TCP, the important distinction is between:

- a **listening socket**, which waits for new clients; and
- a **connected socket**, which carries bytes between one client and
  one server connection.

An IPv4 socket address is a pair `(IP address, port)`. The IP address
identifies a network interface; the port identifies an application
endpoint on that host. We use `127.0.0.1`, the loopback address, so
the client and collector endpoints can both run inside this notebook.

A server establishes its endpoint in three steps:

1. `bind(address)` assigns a local IP address and port to the socket.
2. `listen()` turns it into a listening socket.
3. `accept()` waits for a client and returns a **new connected socket**.

The client calls `connect(server_address)`, asking its operating
system to establish the TCP connection. Once the connection is ready,
the server's `accept` returns. The listening socket remains available
for later clients; the new socket returned by `accept` belongs to
this particular connection.


In [ ]:
import socket

# AF_INET selects IPv4; SOCK_STREAM selects TCP.
listener = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
collector_connection = None

try:
    # Port 0 asks the operating system to choose an unused port.
    listener.bind(("127.0.0.1", 0))
    listener.listen()
    # getsockname() reveals the address assigned to this socket.
    collector_address = listener.getsockname()

    client.connect(collector_address)
    collector_connection, client_address = listener.accept()

    print("Listening address:       ", collector_address)
    print("Client address:          ", client.getsockname())
    print("Accepted peer address:   ", client_address)
    print("Connected server address:", collector_connection.getsockname())
finally:
    client.close()
    if collector_connection is not None:
        collector_connection.close()
    listener.close()


At this point the connection exists, but the operating system knows
nothing about laboratory records, acknowledgements, or clean session
completion. TCP supplies an ordered, reliable transport service. The
application must still define the messages and conversation carried by
that service. That definition is an **application protocol**.


## Part 2 — Observe the byte stream

TCP preserves byte order: if a sender writes `ABCDEFG`, the receiver
obtains those seven bytes in that order. It does **not** promise that
one `recv` call corresponds to one `send` call. A read may return
fewer bytes than requested, or bytes from several writes may already
be waiting together.

The next demonstration makes two writes and then requests the bytes in
differently sized reads.


In [ ]:
support.show_stream_demo()


### Check your understanding

The sender made two calls, one with `ABC` and one with `DEFG`. Can the
receiver infer that boundary from the `recv` results above?

> **Team answer:** Replace this text with one or two sentences.


## Part 3 — Make byte transfer reliable

### Sending all bytes

`sock.send(data)` reports how many bytes the socket accepted. That
number may be smaller than `len(data)`, so an application must retain
an offset into the original data and continue from the first unsent
byte. A return value of zero before completion means the connection
can no longer make progress. If this happens, raise the built-in
`ConnectionError` exception.

Open `ecse316_a1/stream_io.py` and complete `send_all`. Each call to
`send` must offer at most `chunk_size` bytes. Do not replace the loop
with `sendall`; the loop is the point of this exercise. Save the file,
then run the check below.


In [ ]:
support.check_send_all()


### Receiving an exact number of bytes

`sock.recv(n)` means “return **up to** `n` bytes,” not “wait for
exactly `n` bytes.” To recover a fixed-size structure, keep receiving
until the requested count is accumulated. An empty result means EOF;
if bytes are still missing, the stream ended prematurely.

Network sockets exchange bytes rather than Python text strings.
Accordingly, `recv` returns a `bytes` object: `b"ABC"` is three
bytes, whereas `"ABC"` is a Python string. The `b` prefix creates a
bytes literal, and `len(chunk)` tells you how many bytes arrived.

A single call may return only one piece of the data. Since `bytes`
objects are immutable, a convenient pattern is to store the pieces
in a list and join them after all the bytes have arrived:

```python
chunks = [b"AB", b"C", b"DEF"]
combined = b"".join(chunks)
assert combined == b"ABCDEF"
```

Here, `b""` is an empty bytes object used to join the pieces without
placing anything between them. This preserves the original byte
order regardless of how the stream was divided across calls.

In the same file, complete `receive_exactly`. Request at most
`chunk_size` bytes per call, return immediately when `count == 0`,
and raise `UnexpectedEOF` if the peer closes too early. Do not use
`MSG_WAITALL`. Save the file and run the check.


In [ ]:
support.check_receive_exactly()


## Part 4 — Turn the stream into records

**Framing** is the rule an application uses to locate one message
inside a byte stream. The receiver cannot simply read until the
socket is "empty." If no bytes are currently available, a blocking
`recv` waits for more; `recv` returns `b""` only after the peer has
finished sending on the connection.

This protocol keeps the connection open so the client can receive
an acknowledgement and then send another record. Without a record
boundary, the client could wait for an acknowledgement while the
server waited for more bytes. The acknowledgement controls the
order of the exchange; framing tells the server when one record is
complete.

Common framing approaches include:

| Framing method | Idea | Trade-off |
|---|---|---|
| Fixed size | Every message contains the same number of bytes | Simple, but unsuitable for varying record sizes |
| Delimiter | A reserved byte sequence marks the end | Simple when each text message occupies one line; ambiguous if the delimiter also occurs inside the data |
| Length prefix | A header states how many payload bytes follow | Handles arbitrary binary data, but the header must be parsed safely |

Laboratory records vary in size and may contain any byte value, so we
use a length prefix. Our protocol places a fixed 9-byte header before
every payload. The header is metadata that describes how to interpret
the bytes that follow it:

    application record → protocol frame → bytes carried by TCP

    byte offset      0        1                 5                 9
                    +--------+-----------------+-----------------+
                    | type   | payload length  | payload CRC-32  |
                    | 1 byte | 4 bytes         | 4 bytes         |
                    +--------+-----------------+-----------------+
                    | payload bytes ...                           |
                    +---------------------------------------------+


### Message types: what does this frame mean?

The first header byte identifies the purpose of the frame. The
supplied file `ecse316_a1/provided.py` defines `MessageType`, a
Python `IntEnum` with readable names for the three allowed integer
values:

| Python value | Integer carried in the header | Meaning |
|---|---:|---|
| `MessageType.DATA` | 1 | A laboratory record |
| `MessageType.ACK` | 2 | An acknowledgement |
| `MessageType.CLOSE` | 3 | End the protocol conversation |

Use `int(MessageType.DATA)` to obtain the integer placed on the
wire. In the other direction, `MessageType(1)` converts an integer
from a received header back to `MessageType.DATA`. An integer not in
the table, such as 99, raises `ValueError`.

Run the following exploration before continuing.


In [ ]:
from ecse316_a1.provided import MessageType

for message_type in MessageType:
    print(f"{message_type.name:5s} -> header value {int(message_type)}")

decoded_type = MessageType(1)
print("Decoded header value 1 as:", decoded_type.name)

try:
    MessageType(99)
except ValueError as error:
    print("Header value 99 is rejected:", error)


### Packing the header and checking the payload

The diagram above describes the header conceptually. To create those
nine bytes in Python, `ecse316_a1/provided.py` defines this object:

```python
HEADER_STRUCT = struct.Struct("!BII")
```

Python's `struct` module converts integer fields to bytes and back.
In the format `!BII`, `!` selects network byte order, `B` stores the
one-byte message type, and each `I` stores one unsigned four-byte
integer: first the payload length and then the payload CRC-32.
Consequently:

- `HEADER_STRUCT.size` is 9;
- `HEADER_STRUCT.pack(type_value, payload_length, expected_crc)`
  converts those three integer fields into the nine header bytes; and
- `HEADER_STRUCT.unpack(header_bytes)` reverses the operation and
  returns a tuple `(type_value, payload_length, expected_crc)`.

Here, `type_value` is 1, 2, or 3 as introduced above;
`payload_length` is the number of payload bytes following the
header; and `expected_crc` is the checksum placed in the header by
the sender.

The length field tells the receiver exactly how many payload bytes to
accumulate. The supplied function `payload_crc32(payload)` produces
a 32-bit checksum of the payload. The sender stores that number in the
header; the receiver computes it again after receiving the payload.
Different results indicate that the bytes do not match. CRC-32 helps
detect accidental changes, but it is not encryption or protection
against deliberate tampering.

The header format tells us how to represent three integers, but the
protocol also restricts which payload lengths are meaningful:

| Message type | Allowed payload length |
|---|---:|
| `DATA` | From 0 bytes through 1 MiB |
| `ACK` | Exactly 8 bytes |
| `CLOSE` | Exactly 0 bytes |

The next cell uses `HEADER_STRUCT` and `payload_crc32` to construct
one valid `DATA` frame carrying `b"ABC"`.


In [ ]:
from ecse316_a1.provided import (
    HEADER_STRUCT,
    MessageType,
    payload_crc32,
)

example_type = MessageType.DATA
example_payload = b"ABC"
example_length = len(example_payload)
example_crc = payload_crc32(example_payload)

example_header = HEADER_STRUCT.pack(
    int(example_type),
    example_length,
    example_crc,
)
raw_type, decoded_length, decoded_crc = HEADER_STRUCT.unpack(
    example_header
)
decoded_type = MessageType(raw_type)

print("Header size:    ", HEADER_STRUCT.size, "bytes")
print("Header fields:  ", raw_type, decoded_length, f"0x{decoded_crc:08x}")
print("Header bytes:   ", example_header.hex(" "))
print("Complete frame: ", (example_header + example_payload).hex(" "))
print("Decoded type:   ", decoded_type.name)
print("CRC after changing one byte:", f"0x{payload_crc32(b'ABD'):08x}")


### Encoding a frame

Open `ecse316_a1/protocol.py` and find
`encode_frame(message_type, payload)`. This function takes the
purpose of a message and its payload bytes, then returns the complete
header-and-payload byte sequence that can be sent over TCP.

Before creating the header, the function must reject an unknown
message type or a payload length forbidden by the table above.
`ProtocolError`, imported from `provided.py`, is the exception used
for these protocol violations. The function
`validate_payload_length(message_type, length)`, imported from the
same file, checks the three length rules in the table. It returns
normally for a valid pair and raises `ProtocolError` for an invalid
one.

Complete `encode_frame` in this order:

1. Convert `message_type` with `MessageType(message_type)`. Catch an
   unknown value's `ValueError` and raise `ProtocolError` instead.
2. Call `validate_payload_length` with the validated type and
   `len(payload)`.
3. Calculate the checksum with `payload_crc32`.
4. Use `HEADER_STRUCT.pack` to encode the type's integer value, the
   payload length, and the checksum.
5. Return the header bytes followed by the unchanged payload bytes.

Save the file and run the check.


In [ ]:
support.check_encode_frame()


### Sending a frame

Once a frame has been encoded, its bytes can be sent over the TCP
connection. The outgoing path is:

    (message_type, payload) → encode_frame → frame bytes
    frame bytes             → send_all    → TCP stream

Complete `send_frame(sock, message_type, payload, chunk_size)` in
`protocol.py`: encode the frame, then pass all of its bytes to
`send_all` with the given `chunk_size`. Save the file and run the
check.


In [ ]:
support.check_send_frame()


### Recovering a frame

`send_frame` now implements the complete outgoing path. At the other
endpoint, `receive_frame(sock, chunk_size)` implements the incoming
path: it reads one complete frame from the TCP stream, checks it, and
recovers the message type and payload.

Here, `chunk_size` has the same meaning as it did in Part 3: it is
the maximum number of bytes that each call to `sock.recv` may
request. It does not describe the size of a frame.

On success, the function returns `(message_type, payload)`.

Complete `receive_frame` in this order:

1. Use `receive_exactly` and `HEADER_STRUCT.size` to obtain the
   complete header.
2. Unpack its three integers with `HEADER_STRUCT.unpack`.
3. Convert the raw type using `MessageType(raw_type)`, translating an
   unknown value's `ValueError` into `ProtocolError`.
4. Call `validate_payload_length` **before** receiving the payload so
   an illegal or oversized peer-supplied length is rejected first.
5. Use `receive_exactly` again for the declared payload length.
6. Recompute the payload CRC-32 and compare it with the header. Raise
   `ChecksumError` if they differ.
7. Return `(message_type, payload)`.

The supplied exceptions distinguish the possible failures:

- `UnexpectedEOF` when the stream closes before required bytes arrive;
- `ProtocolError` for an unknown type or forbidden length; and
- `ChecksumError` when a complete payload has the wrong CRC-32.

Save `ecse316_a1/protocol.py`, then run the check.


In [ ]:
support.check_receive_frame()


## Part 5 — Define the protocol conversation

A complete application protocol has three parts:

- **syntax:** how messages are encoded—the header and payload format;
- **semantics:** what `DATA`, `ACK`, and `CLOSE` mean; and
- **sequencing:** which messages may follow which earlier messages.

Part 4 established the syntax: we can encode and recover individual
frames. It does not yet tell either endpoint what to do with a
recovered frame. We now add the semantics and sequencing rules.

During one connection, a client repeats the `DATA`/`ACK` exchange
for each of its N records. It waits for each `ACK` before sending the
next record. After record N, it sends `CLOSE` and waits for the
collector server's `CLOSE` response:

    client                    collector (server)
      | ---- DATA(record 1) -----> |
      | <------- ACK ------------- |
      |             ⋮              |
      | ---- DATA(record N) -----> |
      | <------- ACK ------------- |
      | -------- CLOSE ----------> |
      | <------- CLOSE ----------- |

The collector server implements the following state transition
repeatedly:

| Received frame | Collector server action |
|---|---|
| `DATA` | Send `ACK` containing the received length and CRC-32; continue |
| `CLOSE` | Send `CLOSE`; finish the connection |
| `ACK` | Raise `ProtocolError`; clients may not acknowledge the server |


### What goes inside an acknowledgement?

An `ACK` confirms which `DATA` payload the collector server received.
Its payload contains two integers: the number of received bytes and
the CRC-32 of those bytes. Each integer occupies four bytes, so an
`ACK` payload is exactly eight bytes.

`ecse316_a1/provided.py` defines a second `struct` object for this
payload:

```python
ACK_STRUCT = struct.Struct("!II")
```

`ACK_STRUCT.pack(length, checksum)` converts the two integers into
the eight payload bytes. This is not the frame header. After creating
the ACK payload, `send_frame` encodes it with the ordinary nine-byte
header and sends the complete frame.

The following example constructs an acknowledgement for `b"ABC"`.


In [ ]:
from ecse316_a1.provided import ACK_STRUCT, payload_crc32

received_payload = b"ABC"
ack_payload = ACK_STRUCT.pack(
    len(received_payload),
    payload_crc32(received_payload),
)
acknowledged_length, acknowledged_crc = ACK_STRUCT.unpack(ack_payload)

print("ACK payload size:", len(ack_payload), "bytes")
print("ACK payload bytes:", ack_payload.hex(" "))
print("Acknowledged length:", acknowledged_length)
print("Acknowledged CRC-32:", f"0x{acknowledged_crc:08x}")


### Complete the collector server

In Part 1, `accept()` produced a connected socket for one client. A
server normally passes that socket to a **connection handler**: a
function that carries out one complete conversation with that client.
In this assignment, that function is
`handle_client(sock, chunk_size)` in `ecse316_a1/server.py`.

Complete `handle_client` by applying the transition table above:

1. Call `receive_frame` to obtain `(message_type, payload)`.
2. For `DATA`, construct the eight-byte ACK payload with
   `ACK_STRUCT.pack`, send it with `send_frame`, and continue waiting
   for another frame.
3. For `CLOSE`, use `send_frame` to send a `CLOSE` message with an
   empty payload, then return from the handler.
4. For `ACK`, raise `ProtocolError` because the client is not allowed
   to acknowledge the server.

Save `server.py`, then run the next cell. It creates a listening
socket, passes an accepted connection to your handler, and connects a
real TCP client. The client sends a text record and a binary record
containing every byte value from 0 through 255, checks both
acknowledgements, and performs the `CLOSE` exchange.


In [ ]:
demo_acknowledgements = support.run_real_tcp_demo()


## Part 6 — Choosing a record size

Suppose a laboratory instrument has 65,536 bytes ready to deliver.
The application could divide that workload into 1,024 records of 64
bytes, 16 records of 4,096 bytes, or one record of 65,536 bytes. All
three choices carry the same useful data, but this protocol performs
one complete `DATA`/`ACK` exchange for every record.

These are **application records**, not TCP packets. The application
chooses the record boundaries; TCP may divide or combine their bytes
independently.

The experiment asks how record size affects three measurements:

- **first-confirmation latency:** the time from sending the first
  `DATA` frame until its valid `ACK` arrives; and
- **workload completion time:** the time until all 65,536 bytes have
  been acknowledged; and
- **workload goodput:** the rate of useful payload bits delivered
  over that complete transfer.


### Account for the bytes first

Each record adds fixed application-protocol overhead:

- its `DATA` frame contains a 9-byte header; and
- its `ACK` contains another 9-byte header and an 8-byte payload.

A workload divided into $N$ records therefore carries
$65{,}536 + 26N$ application-protocol bytes. Its **protocol
efficiency** is the useful fraction of those bytes:

$$
\text{efficiency}
=
\frac{65{,}536}{65{,}536 + 26N}
$$

This calculation does not include TCP or IP headers. Run the next
cell to calculate the record count, protocol bytes, and efficiency
for each choice.


In [ ]:
from ecse316_a1.provided import ACK_STRUCT, HEADER_STRUCT

experiment_workload_bytes = 65536
experiment_record_sizes = (64, 4096, 65536)
fixed_bytes_per_exchange = 2 * HEADER_STRUCT.size + ACK_STRUCT.size

print(
    "record (B) | records | first DATA frame (B) | "
    "protocol bytes | efficiency (%)"
)
print("-" * 84)
for record_size in experiment_record_sizes:
    records_per_workload = experiment_workload_bytes // record_size
    protocol_bytes = (
        experiment_workload_bytes
        + records_per_workload * fixed_bytes_per_exchange
    )
    efficiency_percent = (
        100 * experiment_workload_bytes / protocol_bytes
    )
    print(
        f"{record_size:10d} | {records_per_workload:7d} | "
        f"{record_size + HEADER_STRUCT.size:20d} | "
        f"{protocol_bytes:14d} | {efficiency_percent:14.3f}"
    )


### Predict performance on a controlled path

The client and server still exchange frames through real loopback
TCP. To make path costs visible, supplied code adds the transmission
and propagation delays of a simple path model: a 10 Mbit/s link and
a 1 ms round-trip time (RTT). Every record size uses the same path.

For a record containing $L$ payload bytes, the modeled path time for
its complete `DATA`/`ACK` exchange is

$$
t_{\text{exchange}}
=
1\text{ ms} + \frac{8(L+26)}{10{,}000{,}000}\text{ s}
$$

Before running the experiment, predict which record size will have
the lowest first-confirmation latency and which will have the highest
workload goodput. Justify both predictions using the table and this
equation.


### Team hypothesis

> **Team answer:** Replace this text with your two predictions and
> reasoning.


### Run the experiment

For each record size, the client transfers the complete 65,536-byte
workload five times, waiting for an `ACK` after every record. Each
condition uses the same workload and path; only record size changes.

The results report the median first-confirmation latency, the median
completion time, and goodput across the five workloads:

$$
\text{goodput}
=
\frac{8 \times 65{,}536 \times 5}
{\text{sum of the five completion times}}
$$


In [ ]:
experiment_link_mbps = 10.0
experiment_round_trip_ms = 1.0
experiment_repetitions = 5

experiment_records = support.run_record_size_experiment(
    workload_bytes=experiment_workload_bytes,
    record_sizes=experiment_record_sizes,
    link_mbps=experiment_link_mbps,
    round_trip_ms=experiment_round_trip_ms,
    repetitions=experiment_repetitions,
)
experiment_summary, experiment_figure = (
    support.display_summary_and_plots(experiment_records)
)
expected_measurements = (
    len(experiment_record_sizes) * experiment_repetitions
)
assert len(experiment_records) == expected_measurements
print(f"Recorded {expected_measurements} measured workloads.")


### Evidence and explanation

> **Team answer:** Replace this text after running the experiment.
> Compare the 64-byte and 65,536-byte record conditions. Cite their
> median first-confirmation latency, completion time, and goodput.
> State whether the evidence supports each part of your hypothesis
> and explain the result using the number and size of the protocol
> exchanges.


## Part 7 — Explain the design

Answer each question briefly.

1. What are the different roles of the listening socket and the
   connected socket returned by `accept()`?

   > **Team answer:** Replace this text.

2. The client waits for an `ACK` before sending another `DATA` frame.
   Why does the protocol still need framing?

   > **Team answer:** Replace this text.

3. Why must `send_all` and `receive_exactly` use loops? What does
   `recv` returning `b""` mean before all requested bytes arrive?

   > **Team answer:** Replace this text.

4. Why does `receive_frame` validate the message type and declared
   length before reading the payload, but check CRC-32 afterward?
   What protection does CRC-32 not provide?

   > **Team answer:** Replace this text.

5. Describe the valid server response to `DATA` and to `CLOSE`. Why
   is receiving an `ACK` from the client a protocol error?

   > **Team answer:** Replace this text.

6. If RTT were negligible, how would record size affect completion
   time and goodput? Why might an application still choose smaller
   application records?

   > **Team answer:** Replace this text.

7. Give one important limitation of the controlled experiment and
   one conclusion that the results do not justify.

   > **Team answer:** Replace this text.


## Final check and submission

Save all three Python files and this notebook. Then choose
**Runtime → Restart session and run all**. Confirm every check passes,
all outputs are saved, and both plots are visible. After Colab shows
that the notebook is saved to Drive, download the complete
`ECSE316_Assignment1` folder from Google Drive. Submit that ZIP through
myCourses.


In [ ]:
support.run_final_checks(
    TEAM_NUMBER,
    TEAM_MEMBERS,
    experiment_records,
)
